# Prompt Engineering + LangChain (via OpenRouter)

**Flow of this notebook:**

1. **Prompt Engineering** — the full prompt engineering tutorial, run directly against **Claude via OpenRouter** (no local model, no GPU needed), with one example cross-checked against **Gemini**.
2. **Basic LangChain Chains** — a minimal prompt → chain → invoke example, using OpenRouter as the LLM backend.
3. **Token Budgeting (tiktoken)** — counting input/output tokens and estimating cost for the calls made above.

> Adapted from the original Colab notebooks (Prompt Engineering tutorial, and *Hands-On Large Language Models* chapter 7). Wherever the original code used a direct OpenAI key/client, it's been swapped for an **OpenRouter** client (OpenAI-compatible API) so the same code can call OpenAI, Gemini, or Anthropic models with one key.

This notebook is entirely API-based — no GPU or local model download required.


## 0. Setup

**Note on the install command below:** package version specifiers like `transformers>=4.41.2` need to be wrapped in quotes when run through `!pip install` in a notebook. Without quotes, the shell reads `>` as an output-redirect operator, silently drops the version constraint, and dumps stray files in your folder instead of actually enforcing it. Quoting (`"package>=1.0"`) avoids that.

**Paste your OpenRouter API key below** (get one at https://openrouter.ai/keys). The same key works across OpenAI, Google, and Anthropic models on OpenRouter.


In [ ]:
!pip install -q "openai>=1.13.3" tiktoken
!pip install -q "langchain>=0.1.17" "langchain-core>=0.1.50" "langchain-openai>=0.1.6" langchain-community


In [ ]:
import os
from openai import OpenAI

# --- OpenRouter client (OpenAI-compatible API) ---
# Paste your OpenRouter key here, or set it as an environment variable beforehand.
OPENROUTER_API_KEY = "PASTE_YOUR_OPENROUTER_API_KEY_HERE"
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

def chat(model, messages, **kwargs):
    """Call any OpenRouter-hosted model (OpenAI / Gemini / Anthropic / etc.) with the same interface."""
    response = client.chat.completions.create(model=model, messages=messages, **kwargs)
    return response.choices[0].message.content


## Part 1: Prompt Engineering

*(Adapted from `Tutorial_2_Prompt_Engineering_ver2_1_.ipynb`. The original ran everything on a locally-loaded Phi-3 model via a Hugging Face `pipeline`. It now runs directly on **Claude via OpenRouter** using the `chat()` helper from Setup, with explicit `temperature` / `top_p` on each call. One example is also cross-checked against **Gemini** for comparison.)*


### A note on temperature and top_p over the API

The original tutorial's temperature/top_p demos were tuned for a local model, where temperature can go above 1 (it used up to 1.9). Claude's API only accepts **temperature 0–1** and **top_p 0–1**, so the "high temperature" examples below use `1.0` (its max) instead of `1.9` — the contrast between "low" and "high" still comes through, just within Claude's valid range. There's no `do_sample` parameter with a hosted API — sampling always happens; `temperature=0` is the closest thing to deterministic.


In [ ]:
import textwrap

CLAUDE_MODEL = "anthropic/claude-sonnet-4.5"  # see openrouter.ai/models for the current list


# 1. Zero-shot Prompting: Basic Dish Recommendation

## 1.1 Declaring the Prompt and Running through the pipeline to generate output

In [ ]:
# Prompt
messages = [
    {"role": "user", "content": "Recommend a dish to someone who is hungry and likes spicy food."}
]

# Generate the output directly via OpenRouter
output = chat(model=CLAUDE_MODEL, messages=messages)
print(textwrap.fill(output, width=50))


### 1.1.1 Same prompt via OpenRouter — Gemini (cross-provider comparison)

The cell above already runs on Claude. This cell runs the exact same prompt on a different OpenRouter model (Gemini) so you can compare outputs across providers.


In [ ]:
# Same zero-shot prompt, run via OpenRouter on Gemini for comparison
gemini_output = chat(
    model="google/gemini-2.5-flash",  # see openrouter.ai/models for the current list
    messages=messages,
)
print(gemini_output)


## 1.2 Using the apply_chat_template method to convert structured chat messages into a plain text prompt

In [ ]:
# With a hosted API model like Claude, there's no local tokenizer / chat-template step —
# you just send the structured `messages` list directly and the API handles formatting
# internally. (The original tutorial needed apply_chat_template() because it was building
# a raw text prompt for a local model.)
messages = [
    {"role": "user", "content": "Recommend a dish to someone who is hungry and likes spicy food."}
]
print(messages)


##1.3 Arguments like temperature and top_p while running the pipeline
### 1.3.1 temperature controls how creative or random the model is when choosing words.


In [ ]:
# Claude's temperature range is 0-1 (the original tutorial used up to 1.9 for a local model);
# 1.0 is the highest value the API accepts.
output = chat(model=CLAUDE_MODEL, messages=messages, temperature=1.0)
print(textwrap.fill(output, width=100))


### 1.3.2 top-p chooses from the smallest group of words that together have p% of the probability.

In [ ]:
output = chat(model=CLAUDE_MODEL, messages=messages, top_p=0.5)
print(textwrap.fill(output, width=100))


### 1.3.3 High Temperature and High p gives creative and unexpected outputs among the large pool of potential tokens

In [ ]:
messages = [
    {"role": "user", "content": "What are some unconventional ways to increase weekday lunch traffic at my restaurant?"}
]

output = chat(model=CLAUDE_MODEL, messages=messages, temperature=1.0, top_p=1)
print(textwrap.fill(output, width=100))


### 1.3.4 Low Temperature and Low p gives deterministic output that are predictable and focused. No creativity and no surprises.

In [ ]:
messages = [
    {"role": "user", "content": "Write a message to staff explaining how to handle customer complaints politely and professionally"}
]

output = chat(model=CLAUDE_MODEL, messages=messages, temperature=0.1, top_p=0.1)
print(textwrap.fill(output, width=100))


### 1.3.5 Low Temperature and High p gives deterministic output that have a bit more variety. Outputs are deterministic but with highly probable predicted tokens

In [ ]:
messages = [
    {"role": "user", "content": "Write short, appealing descriptions for each dish on our dinner menu"}
]

output = chat(model=CLAUDE_MODEL, messages=messages, temperature=0.1, top_p=1)
print(textwrap.fill(output, width=100))


### 1.3.6 High Temperature and Low p gives creative outputs but within a predictable pool of tokens

In [ ]:
messages = [
    {"role": "user", "content": "Give creative but clear names for our classic dishes like grilled chicken, Caesar salad, and chocolate cake"}
]

output = chat(model=CLAUDE_MODEL, messages=messages, temperature=1.0, top_p=0.1)
print(textwrap.fill(output, width=100))


##1.4 Assigning Roles in Prompt and declaring Dynamic Prompts

In [ ]:
# Indian Restaurant Review Summarization Prompt

# Sample text (replace this with any real or synthetic review text)
text = """Masala Mahal is a vibrant Indian restaurant located in the heart of the city. The décor blends traditional Indian elements with a modern touch — brass lamps, colorful murals, and soft sitar music set the mood.
The food is authentic and flavorful. Highlights included the butter chicken, which was creamy and well-spiced, and the paneer tikka, grilled to perfection. The garlic naan was fluffy and complemented the curries beautifully.
There were plenty of vegetarian and vegan options, with a separate Jain menu available. Service was courteous and prompt, though during peak hours there was a slight delay in getting appetizers.
The desserts, especially the gulab jamun, were warm and satisfying. Overall, Masala Mahal offers a delightful experience with traditional Indian hospitality."""

# Prompt components
persona = (
    "You are ChefBot, a virtual restaurant assistant created for Indian restaurants. "
    "You specialize in analyzing customer feedback and summarizing it into actionable insights for chefs and staff.\n"
)

instruction = "Summarize the customer review provided.\n"

context = (
    "Your summary should highlight important aspects of the dining experience, especially focusing on food quality, service, ambiance, variety, and any concerns or complaints.\n"
)

data_format = (
    "Format your summary as bullet points under the following categories: "
    "Food, Ambiance, Service, Variety, Drawbacks. "
    "End with a brief paragraph summarizing the customer's overall impression.\n"
)

audience = (
    "The summary is for the restaurant owner and kitchen staff who need to quickly understand what the customer appreciated or disliked.\n"
)

tone = "The tone should be professional, clear, and actionable.\n"

data = f"Customer review to summarize:\n{text}"

# Final Prompt
query = persona + instruction + context + data_format + audience + tone + data


In [ ]:
messages = [
    {"role": "user", "content": query}
]

# No local chat template needed for a hosted API model — just view the composed prompt text
print(textwrap.fill(messages[0]["content"], width=100))


In [ ]:
# Generate the output directly via OpenRouter
output = chat(model=CLAUDE_MODEL, messages=messages)
print(textwrap.fill(output, width=100))


## 1.5 In-Context Learning: Trying to recognise if the dish is Vegan or not

In [ ]:
one_shot_prompt = [
    {
        "role": "user",
        "content": (
            "Determine if the following dish is vegan-friendly or not.\n\n"
            "Dish: Paneer Butter Masala\n"
            "Description: Indian cottage cheese cubes cooked in a creamy tomato-based gravy with butter and cream.\n"
            "Is it vegan?"
        )
    },
    {
        "role": "assistant",
        "content": "❌ No, it contains dairy (paneer, butter, and cream)."
    },
    {
        "role": "user",
        "content": (
            "Dish: Chana Masala\n"
            "Description: Chickpeas simmered in a spicy onion-tomato gravy with garlic, ginger, and traditional spices.\n"
            "Is it vegan?"
        )
    },
    {
        "role": "assistant",
        "content": "✅ Yes, it is typically vegan as it contains no animal products."
    },
    {
        "role": "user",
        "content": (
            "Dish: Vegetable Biryani\n"
            "Description: Aromatic rice cooked with mixed vegetables, saffron, and spices. Often served with raita.\n"
            "Is it vegan?"
        )
    }
]

# No local chat template needed — just look at the structured messages list directly
print(one_shot_prompt)


In [ ]:
# Generate the output directly via OpenRouter
output = chat(model=CLAUDE_MODEL, messages=one_shot_prompt)
print(textwrap.fill(output, width=100))


## 1.6 Chain Prompting: Coming up with new creative dish names

In [ ]:
# Generate a name and slogan for a new Indian fusion dish

dish_prompt = [
    {
        "role": "user",
        "content": "Create a name and slogan for a spicy Indian street food dish made with crispy tofu, tangy coriander chutney, and chili flakes."
    }
]

# Generate the output directly via OpenRouter
dish_description = chat(model=CLAUDE_MODEL, messages=dish_prompt)

print(textwrap.fill(dish_description, width=100))


In [ ]:
# Step 2: Use the generated name and slogan in a follow-up prompt
pitch_prompt = [
    {
        "role": "user",
        "content": f"""Using the following dish name and slogan, write a short, engaging pitch that a waiter or voice assistant could say to a customer:

{dish_description}

Make it sound friendly, mouth-watering, and tempting, in under 60 words."""
    }
]

pitch_text = chat(model=CLAUDE_MODEL, messages=pitch_prompt)

print(textwrap.fill(pitch_text, width=100))


# 2 Reasoning with Generative Models

## 2.1 Chain-of-Thought: Think Before Answering

In [ ]:
# Answering kitchen inventory questions without showing reasoning

kitchen_prompt = [
    {
        "role": "user",
        "content": "We have 10 bags of basmati rice. We ordered 5 more today. How many bags do we have now?"
    },
    {
        "role": "assistant",
        "content": "15"
    },
    {
        "role": "user",
        "content": "There are 30 mangoes in the kitchen. The chef used 12 for smoothies and we received 10 more from the supplier. How many mangoes are there now?"
    }
]

# Run the model directly via OpenRouter
output = chat(model=CLAUDE_MODEL, messages=kitchen_prompt)
print(output)


In [ ]:
# Answering with Chain-of-Thought (Step-by-step reasoning)

chefbot_cot_prompt = [
    {
        "role": "user",
        "content": "We have 10 kg of rice. We used 4 kg to prepare biryani. Then we received 7 kg more in the new delivery. How much rice do we have now?"
    },
    {
        "role": "assistant",
        "content": "We started with 10 kg of rice. 4 kg was used for biryani, so we have 6 kg left. Then we received 7 kg more. 6 + 7 = 13 kg. The answer is 13 kg."
    },
    {
        "role": "user",
        "content": "There were 40 samosas in the kitchen. 25 were sold at lunch, and 15 more were prepared in the evening. How many samosas are there now?"
    }
]

# Run the model directly via OpenRouter
output = chat(model=CLAUDE_MODEL, messages=chefbot_cot_prompt)
print(output)


## 2.3 Zero-shot Chain-of-Thought

In [ ]:
#Zero-Shot Chain-of-Thought (no examples, just "Let's think step-by-step")

chefbot_zeroshot_cot_prompt = [
    {
        "role": "user",
        "content": (
            "We had 15 liters of milk in the fridge. The chef used 9 liters for tea and desserts. "
            "Then we received 5 more liters in the morning delivery. How much milk do we have now? Let's think step-by-step."
        )
    }
]

# Run the model directly via OpenRouter
output = chat(model=CLAUDE_MODEL, messages=chefbot_zeroshot_cot_prompt)
print(output)


# 3 Tree-of-Thought: Exploring Intermediate Steps

In [ ]:
zeroshot_tot_prompt = [
    {
        "role": "user",
        "content": (
            "Imagine three different kitchen assistants are answering this question. "
            "All assistants will write down each step of their thinking, then share it with the group. "
            "Then all assistants will go on to the next step, etc. If any assistant realizes they're wrong at any point, they leave. "
            "The question is: 'We had 18kg of onions. 10kg were used for curry prep, and 4kg arrived in the morning delivery. How many kilograms of onions are there now?' "
            "Make sure to print each step as assistants discuss and agree on the final result."
        )
    }
]


In [ ]:
# Generate the output directly via OpenRouter
output = chat(model=CLAUDE_MODEL, messages=zeroshot_tot_prompt)
print(output)


# 4 Output Verification

## 4.1 Zero-Shot prompt to create a dish profile in JSON

In [ ]:
chefbot_zeroshot_prompt = [
    {
        "role": "user",
        "content": "Create a character profile for a new Indian seven course meal in JSON format."
    }
]

# Run the model directly via OpenRouter
output = chat(model=CLAUDE_MODEL, messages=chefbot_zeroshot_prompt)
print(output)


## 4.2 One Shot Prompt providing Output Structure Example

In [ ]:
one_shot_template = """Create a short dish profile for an Indian restaurant menu. Use only this format:

{
  "name": "DISH NAME",
  "ingredients": ["INGREDIENT 1", "INGREDIENT 2", "..."],
  "spice_level": "MILD, MEDIUM, or HOT",
  "description": "A SHORT DESCRIPTION OF THE DISH"
}
"""

one_shot_prompt = [
    {"role": "user", "content": one_shot_template}
]

# Run the model directly via OpenRouter
output = chat(model=CLAUDE_MODEL, messages=one_shot_prompt)
print(output)


# 5. Structured JSON Output

The original tutorial's "Grammar: Constrained Sampling" section used `llama-cpp-python`'s local grammar-constrained decoding to force syntactically valid JSON, token by token, from a locally-loaded model. That's a local-inference-time technique — a hosted chat API like Claude doesn't expose token-level grammar constraints, so it isn't a like-for-like swap. The closest equivalent via Claude is asking directly for JSON and parsing the response, shown below.


In [ ]:
# Ask Claude for JSON directly. Note this is prompt-based JSON, not grammar-constrained
# sampling — true grammar-constrained decoding only exists for local inference engines
# like llama.cpp, not hosted chat APIs.
json_prompt = [
    {"role": "user", "content": "Create a menu for a Corporate Party. Respond with ONLY valid JSON, no other text, no markdown code fences."}
]

output = chat(model=CLAUDE_MODEL, messages=json_prompt)


In [ ]:
import json

# Claude (like most chat models) will sometimes wrap JSON in a ```json ... ``` code fence
# or add a stray sentence even when told not to. Pull out just the {...} object before parsing
# instead of relying on the response being pure JSON.
start = output.find("{")
end = output.rfind("}") + 1
json_output = json.dumps(json.loads(output[start:end]), indent=4)
print(json_output)


## Part 2: Basic LangChain Chains

*(Adapted from `Chapter_7_-_Advanced_Text_Generation_Techniques_and_Tools.ipynb`. The original used a local `LlamaCpp` model / a raw `ChatOpenAI(... os.environ["OPENAI_API_KEY"] ...)` call for the agent section; here the chain's LLM is `ChatOpenAI` pointed at OpenRouter, so it can call any OpenRouter model with the same OpenAI-style client.)*


In [ ]:
from langchain_openai import ChatOpenAI

# ChatOpenAI pointed at OpenRouter instead of directly at OpenAI
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",  # any OpenRouter model string works here
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0,
)


In [ ]:
from langchain_core.prompts import PromptTemplate

# Create a prompt template with the "input_prompt" variable
template = "{input_prompt}"
prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)

# Basic chain: prompt -> llm
basic_chain = prompt | llm


In [ ]:
# Use the chain
basic_chain.invoke(
    {
        "input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
    }
)


## Part 3: Token Budgeting (tiktoken)

Estimating input/output token counts and cost for the LLM calls made above. `tiktoken` is OpenAI's tokenizer, so counts for Gemini/Claude are an *approximation* (each provider tokenizes slightly differently) — good enough for budgeting, not exact billing.


In [ ]:
import tiktoken

# cl100k_base is the encoding used by GPT-3.5/GPT-4-era OpenAI models;
# it's used here as a reasonable general-purpose approximation for any model.
encoding = tiktoken.get_encoding("cl100k_base")

def estimate_cost(prompt_text, completion_text, model_name="openai/gpt-4o-mini",
                   input_price_per_1k=0.00015, output_price_per_1k=0.0006):
    """
    Rough token count + cost estimate for a single prompt/completion pair.
    Price defaults are illustrative — check OpenRouter's pricing page for the model you use.
    """
    input_tokens = len(encoding.encode(prompt_text))
    output_tokens = len(encoding.encode(completion_text))

    input_cost = (input_tokens / 1000) * input_price_per_1k
    output_cost = (output_tokens / 1000) * output_price_per_1k

    print(f"Model: {model_name}")
    print(f"Input tokens : {input_tokens}  (~${input_cost:.6f})")
    print(f"Output tokens: {output_tokens}  (~${output_cost:.6f})")
    print(f"Total cost   : ~${input_cost + output_cost:.6f}")

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "estimated_cost_usd": input_cost + output_cost,
    }


In [ ]:
# Example: budget the Gemini call from Part 1 (section 1.1.1)
_ = estimate_cost(
    prompt_text=messages[0]["content"] if isinstance(messages, list) else str(messages),
    completion_text=gemini_output,
    model_name="google/gemini-2.5-flash",
)
